In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean
import re 
import ast

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

Generating the dataset

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" , "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""


    messages = []
    add_user_message(messages , prompt)
    add_assistant_message(messages , "```json")
    text = chat(messages , stop_sequences=["```"])
    return json.loads(text)

In [ ]:
def run_prompt(test_case):
    "Merges the prompt and test case input , then return the result"
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with python, JSON, or a plain Regex
* Do not add any comments or comentary or explanation
    """

    messages = []
    add_user_message(messages , prompt)
    add_assistant_message(messages , "```code")   # prefilling messages
    output = chat(messages, stop_sequences = ['```'])
    return output

Model grading

In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

Code grading

In [ ]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
def run_test_case(test_case):
    "Calls run_prompt, then grades the result"
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case , output)
    model_score = model_grade['score']
    reasoning = mode_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output" : output,
        "test_case" : test_case,
        "score" : score,
        "reasoning" : reasoning,

    }

In [ ]:
def run_eval(dataset):
    "Loads the dataset and calls the run_test_case with each case"
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    
    return results